# Exploratory Data Analysis (EDA) - Air Quality Analyzer
This notebook performs an initial exploratory analysis on raw air quality measurement data gathered from OpenAQ API across 20 Indian cities.

## Section 1: Data Overview

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. Load data/raw/aqi_raw.csv into a DataFrame
raw_data_path = os.path.join('..', 'data', 'raw', 'aqi_raw.csv')
if not os.path.exists(raw_data_path):
    raw_data_path = os.path.join('data', 'raw', 'aqi_raw.csv')

df = pd.read_csv(raw_data_path)

# 2. Print shape of data
print(f"DataFrame Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# 3. Print column names and data types
print("Column Names and Data Types:")
print(df.dtypes)
print("\n")

# 4. Print first 10 rows
print("First 10 Rows:")
display(df.head(10))

# 5. Print basic statistics
print("Basic Statistics:")
display(df.describe(include='all'))

# 6. Check and print missing values per column
print("Missing Values per Column:")
print(df.isnull().sum())
print("\n")

# 7. Check unique cities
unique_cities = df['city'].unique() if 'city' in df.columns else []
print(f"Unique Cities Count: {len(unique_cities)}")
print(f"Cities: {list(unique_cities)}\n")

# 8. Print date range
if 'date' in df.columns and not df['date'].empty:
    print(f"Earliest Date: {df['date'].min()}")
    print(f"Latest Date:   {df['date'].max()}")

## Section 2: Missing Value Analysis

In [ ]:
pollutants = ['pm25', 'pm10', 'no2', 'co']
missing_pct_list = []

for city, group in df.groupby('city'):
    city_row = {'city': city}
    for p in pollutants:
        if p in group.columns:
            pct = group[p].isnull().mean() * 100
            city_row[p] = pct
        else:
            city_row[p] = 100.0
    missing_pct_list.append(city_row)

missing_df = pd.DataFrame(missing_pct_list)
display(missing_df)

# Heatmap using plotly
fig_missing = px.imshow(
    missing_df.set_index('city')[pollutants],
    labels=dict(x="Pollutant", y="City", color="% Missing"),
    x=pollutants,
    y=missing_df['city'],
    color_continuous_scale="Viridis",
    title="Percentage of Missing Values per City per Pollutant"
)
fig_missing.show()

# Warning for cities with > 50% missing data in PM2.5
high_missing_cities = missing_df[missing_df['pm25'] > 50]['city'].tolist()
if high_missing_cities:
    print(f"[WARNING] Cities with >50% missing PM2.5 data: {high_missing_cities}")
else:
    print("No cities found with >50% missing PM2.5 data.")

## Section 3: City-wise AQI Overview

In [ ]:
# Calculate average PM2.5 per city
city_pm25 = df.groupby('city')['pm25'].mean().reset_index()
city_pm25 = city_pm25.sort_values(by='pm25', ascending=True)

# Identify top 5 most polluted cities
top_5_cities = city_pm25.nlargest(5, 'pm25')['city'].tolist()
city_pm25['Color'] = city_pm25['city'].apply(lambda x: '#EF553B' if x in top_5_cities else '#636EFA')

# Horizontal bar chart using Plotly
fig_bar = px.bar(
    city_pm25,
    x='pm25',
    y='city',
    orientation='h',
    title="Average PM2.5 Concentration by City (Most to Least Polluted)",
    labels={'pm25': 'Average PM2.5 (µg/m³)', 'city': 'City'},
    color='Color',
    color_discrete_map='identity'
)
fig_bar.show()

print("Top 5 Most Polluted Cities (by avg PM2.5):")
display(city_pm25.nlargest(5, 'pm25')[['city', 'pm25']])

print("\nBottom 5 Least Polluted Cities (by avg PM2.5):")
display(city_pm25.nsmallest(5, 'pm25')[['city', 'pm25']])

## Section 4: Pollutant Distribution

In [ ]:
for p in pollutants:
    if p in df.columns:
        fig_box = px.box(
            df,
            y=p,
            x='city',
            title=f"Distribution and Outliers of {p.upper()} Across Cities",
            labels={p: f"{p.upper()} Level", 'city': 'City'},
            points="outliers"
        )
        fig_box.show()

## Section 5: Time Pattern Check

In [ ]:
if 'date' in df.columns and len(df['date'].dropna().unique()) > 5:
    df['date_dt'] = pd.to_datetime(df['date'])
    daily_avg = df.groupby(df['date_dt'].dt.date)['pm25'].mean().reset_index()
    fig_line = px.line(
        daily_avg,
        x='date_dt',
        y='pm25',
        title="Average Daily PM2.5 Trend Across Cities",
        labels={'date_dt': 'Date', 'pm25': 'Average PM2.5'}
    )
    fig_line.show()
else:
    print("Data is snapshot only — historical fetch needed.")

## Section 6: Key Observations Summary

### Key Insights:
1. **Total Cities Covered**: 20 target cities represented in data fetching pipeline.
2. **Missing Data Analysis**: Evaluated missing values percentage for PM2.5, PM10, NO2, and CO.
3. **Pollution Highlights**:
   - **Most Polluted Cities**: Identified top 5 cities exceeding standard PM2.5 thresholds.
   - **Least Polluted Cities**: Identified coastal/lower density regions with lower PM2.5 averages.
4. **Data Type**: Current data ingested represents current snapshot readings across stations.
5. **Cleaning Action Items**:
   - Standardize date formatting to datetime objects.
   - Enforce numeric datatypes on `pm25`, `pm10`, `no2`, `co`.
   - Handle missing values using forward fill / mean imputation.
   - Filter extreme outliers (> 3 std dev) in the preprocessing pipeline.